# NLP Lab 4 Task

### **Instructions for Students:**
1. All data preparation, vocabularies, and boilerplate loader functions are pre-written.
2. Do **NOT** copy-paste standard models from the lab demo. Read the specific architectural or algorithmic twists for each task carefully.
3. You must replace all `raise NotImplementedError()` blocks with your code implementation.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import gensim.downloader as api
import numpy as np
import random
import urllib.request
import re

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load Pre-trained Google Word2Vec (300-dim)
print("Loading Google Word2Vec Model...")
w2v_google = api.load('word2vec-google-news-300')

def build_embedding_matrix(corpus, w2v_model):
    vocab = {"<PAD>": 0, "<UNK>": 1}
    embedding_dim = w2v_model.vector_size
    weights = [np.zeros(embedding_dim), np.random.randn(embedding_dim) * 0.1]
    
    for sentence in corpus:
        for word in sentence:
            if word not in vocab:
                vocab[word] = len(vocab)
                if word in w2v_model:
                    weights.append(w2v_model[word])
                else:
                    weights.append(np.random.randn(embedding_dim) * 0.1)
                    
    return vocab, torch.tensor(np.array(weights), dtype=torch.float32)

# Autoregressive Language Modeling with Top-$K$ Sampling & Perplexity
### **The Twist / Challenge:**
Standard greedy decoding or unconstrained sampling can produce repetitive or nonsensical tokens from the long tail of the probability distribution. In this task, you will implement an autoregressive Word-Level Stacked LSTM Language Model trained on the TinyShakespeare corpus with two key challenges:
1. Top-$K$ Temperature Sampling: Modify the generation loop to:
- Scale the output logits by a temperature parameter $T$.
- Identify the top $K$ largest logit values using torch.topk.
- Mask out all non-top-$K$ logits by setting them to $-\infty$ (-float('Inf')).
- Compute softmax over the masked logits and sample the next token with torch.multinomial.
2. Perplexity Metric: Implement the model evaluation function to compute Perplexity ($PPL$) over a validation set:$$PPL = \exp(\text{CrossEntropyLoss})$$

In [ ]:
import urllib.request
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split

# 1. Download TinyShakespeare Corpus
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
raw_text = urllib.request.urlopen(url).read().decode('utf-8')[:30000]

# Word-level tokenization (lowercased)
words = re.findall(r"\b\w+\b", raw_text.lower())

# 2. Build Vocabulary and 300-dim Google Word2Vec Embedding Matrix
chunk_size = 50
word_chunks = [words[i:i + chunk_size] for i in range(0, len(words), chunk_size)]
vocab_t4, embed_matrix_t4 = build_embedding_matrix(word_chunks, w2v_google)
idx2word_t4 = {v: k for k, v in vocab_t4.items()}

# 3. Build Word-Level Sequence Targets (Sliding Window)
seq_length = 6
lm_X, lm_Y = [], []
encoded_words = [vocab_t4.get(w, vocab_t4["<UNK>"]) for w in words]

for i in range(0, len(encoded_words) - seq_length):
    lm_X.append(encoded_words[i : i + seq_length])
    lm_Y.append(encoded_words[i + 1 : i + seq_length + 1])

X_lm_tensor = torch.tensor(lm_X, dtype=torch.long)
Y_lm_tensor = torch.tensor(lm_Y, dtype=torch.long)

# Split into Train (80%) and Validation (20%) Sets
full_dataset = TensorDataset(X_lm_tensor, Y_lm_tensor)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size], 
    generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

print(f"TinyShakespeare Tokens: {len(words)}")
print(f"Vocabulary Size: {len(vocab_t4)}")
print(f"Train Sequences: {len(train_dataset)} | Validation Sequences: {len(val_dataset)}")

In [ ]:
# ==============================================================================
# TODO 4.1: Complete Stacked 2-Layer LSTM Language Model
# ==============================================================================

class StackedLSTMLanguageModel(nn.Module):
    def __init__(self, embed_matrix, hidden_dim=128, num_layers=2):
        super().__init__()
        # TODO 1: Initialize pretrained Google Word2Vec embedding layer using nn.Embedding.from_pretrained
        # TODO 2: Initialize multi-layer nn.LSTM (num_layers=num_layers, batch_first=True)
        # TODO 3: Initialize Linear classifier layer mapping hidden_dim -> vocab_size
        vocab_size, embed_dim = embed_matrix.shape
        self.embedding = nn.Embedding.from_pretrained(embed_matrix, freeze=False, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)
        
    def forward(self, x, hidden=None):
        """
        x: (batch_size, seq_len)
        hidden: optional tuple (h_0, c_0) for autoregressive generation
        """
        # TODO 1: Pass x through embedding layer
        # TODO 2: Pass embedded representations through self.lstm
        # TODO 3: Project outputs through self.fc to compute logits across vocabulary
        # Return: logits (batch_size, seq_len, vocab_size), hidden_states
        embeds = self.embedding(x)
        out, hidden = self.lstm(embeds, hidden)
        logits = self.fc(out)
        return logits, hidden


# ==============================================================================
# TODO 4.2: Complete Autoregressive Generation with Top-K Temperature Sampling
# ==============================================================================

def generate_top_k_shakespeare(model, prompt="first citizen", k=5, temperature=0.8, gen_len=20):
    model.eval()
    prompt_words = re.findall(r"\b\w+\b", prompt.lower())
    input_indices = [vocab_t4.get(w, vocab_t4["<UNK>"]) for w in prompt_words]
    input_eval = torch.tensor([input_indices], dtype=torch.long).to(device)
    
    generated_words = list(prompt_words)
    hidden = None
    
    with torch.no_grad():
        for _ in range(gen_len):
            # TODO 1: Forward pass input_eval and hidden state through model
            # TODO 2: Extract logits for the last token position -> shape: (1, vocab_size)
            # TODO 3: Scale logits by temperature: logits = logits / temperature
            # TODO 4: Find top k values and indices using torch.topk(logits, k, dim=-1)
            # TODO 5: Create a new tensor filled with -float('Inf') matching logits shape
            # TODO 6: Scatter the top-k values into the masked tensor
            # TODO 7: Apply F.softmax to masked logits to obtain probabilities
            # TODO 8: Sample 1 token index using torch.multinomial(probs, num_samples=1)
            # TODO 9: Append sampled token word to generated_words and update input_eval tensor
            logits, hidden = model(input_eval, hidden)
            last_logits = logits[:, -1, :]
            scaled_logits = last_logits / temperature
            topk_values, topk_indices = torch.topk(scaled_logits, k, dim=-1)
            masked_logits = torch.full_like(scaled_logits, -float('Inf'))
            masked_logits.scatter_(dim=-1, index=topk_indices, src=topk_values)
            probs = F.softmax(masked_logits, dim=-1)
            sampled_idx = torch.multinomial(probs, num_samples=1)
            next_word = idx2word_t4.get(sampled_idx.item(), "<UNK>")
            generated_words.append(next_word)
            input_eval = sampled_idx
            
    return " ".join(generated_words)


# ==============================================================================
# TODO 4.3: Complete Validation Perplexity Calculation Function
# ==============================================================================

def calculate_perplexity(model, val_loader, criterion):
    model.eval()
    total_loss = 0.0
    total_batches = 0
    vocab_size = len(vocab_t4)
    
    with torch.no_grad():
        for bx, by in val_loader:
            bx, by = bx.to(device), by.to(device)
            # TODO 1: Forward pass through model to get logits
            # TODO 2: Compute loss using criterion (reshape logits to (-1, vocab_size) and targets to (-1))
            # TODO 3: Accumulate loss over all validation batches
            logits, _ = model(bx)
            loss = criterion(logits.view(-1, vocab_size), by.view(-1))
            total_loss += loss.item()
            total_batches += 1
            
    avg_loss = total_loss / total_batches
    # TODO 4: Compute perplexity PPL = exp(avg_loss) and return float value
    return float(np.exp(avg_loss))


# ==============================================================================
# Test Execution Block for Verification
# ==============================================================================

# lm_model = StackedLSTMLanguageModel(embed_matrix_t4, hidden_dim=128, num_layers=2).to(device)
# lm_criterion = nn.CrossEntropyLoss(ignore_index=0)
# lm_optimizer = optim.Adam(lm_model.parameters(), lr=0.003)

# Train model for a few epochs...
# val_ppl = calculate_perplexity(lm_model, val_loader, lm_criterion)
# sample_text = generate_top_k_shakespeare(lm_model, prompt="to be or not", k=5, temperature=0.7, gen_len=25)

# print(f"Validation Perplexity (PPL): {val_ppl:.2f}")
# print(f"Generated Text:\n{sample_text}")